# Tracing quickstart — seeing inside an LLM call

*Part of the `gen_ai/` track. Prerequisites: `basics/a_setup_mlflow` (a running tracking server) and `basics/b_tracking_quickstart` (experiments & runs).*

## The problem: a GenAI call isn't a flat record

In the `ml/` track a run is flat: parameters in, metrics out.   
A **GenAI** request isn't flat — one user question fans out into steps:

- build a prompt, 
- call the model, 
- maybe call a tool, 
- parse the result. 

When the answer is wrong, a params-and-metrics view can't tell you *which step* misbehaved or *how long* each took.

### What is a `span`
**MLflow Tracing** records each step as a **span** — its inputs, outputs, latency, and (for LLM calls) token usage. 

### What is a `trace`
The spans of one request form a **trace**. This is the **Traces** tab you saw sitting empty in `basics/a_setup_mlflow` — this notebook is what finally fills it.

| Term | One-line meaning |
|---|---|
| **span** | One unit of work — a function call or an LLM call — with its inputs, outputs, start/end time. |
| **trace** | The tree of spans produced by handling one request. |
| **automatic tracing** | One line — `mlflow.openai.autolog()` — that records a span for every LLM client call, with no code changes. |

## Prerequisites

**1. A tracking server on port 5001** (the advanced-track port, shared with `ml/`). From `src/`, in a separate terminal:

```bash
mlflow server --host 127.0.0.1 --port 5001
```

**2. A local LLM via [Ollama](https://ollama.com).** Ollama is a local model runtime that exposes an **OpenAI-compatible API** at `http://localhost:11434/v1` — so the standard `openai` client talks to it with no key and no cost. Install it (see the download page), then pull a model and make sure it's serving:

```bash
ollama pull qwen3:8b     # the model this notebook uses
ollama serve             # usually already running as a background service
```

This notebook uses **`qwen3:8b`** (about a 5 GB download). Ollama runs the model on your GPU when it fits and falls back to the CPU otherwise, so it works with or without a GPU — just slower on CPU. Match the size to your machine: **`gemma3:4b`** or **`qwen3:4b`** are lighter and faster; **`qwen3:14b`** is stronger if you have the memory for it. Any chat model works — set `MODEL` in the code below to whatever you pulled.

**3. The `openai` client.** It's already a dependency of this repo, so `uv sync` installs it. (Working outside the repo? `uv add openai` — the same client talks to both Ollama and OpenAI.)

**Diverges from upstream tutorial:** MLflow's tracing quickstarts call the **OpenAI API**, which needs a paid key. This repo defaults to **local Ollama** through its OpenAI-compatible endpoint so the tutorial runs at zero cost; the one-line swap to the real OpenAI API is shown at the end.

## Turn on tracing

Point the client at the tracking server, name an experiment, and enable automatic tracing. `mlflow.openai.autolog()` *patches* the `openai` client so that, from here on, every `chat.completions.create(...)` call records a span automatically — no logging code in your app.

**Why `openai` autolog if we're calling Ollama?** Autolog instruments the **client library**, not the service behind it. We reach Ollama through the **`openai` client** (its OpenAI-compatible API), so patching that client is exactly what captures the call — the `base_url` it points at doesn't matter. There is no `mlflow.ollama.autolog()`: Ollama isn't a separate MLflow integration, you trace it through whichever client you call it with (here `openai`; it could just as well be LangChain or LlamaIndex). If you instead used the *native* `ollama` Python package, autolog wouldn't see it — you'd fall back to the manual `@mlflow.trace` shown later in this notebook.

In [ ]:
import mlflow
from openai import OpenAI

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-tracing-quickstart")

# One line: trace every call made with the `openai` client (Ollama included).
mlflow.openai.autolog()

## A note on Qwen3's thinking mode

`qwen3:8b` is a *reasoning* model: by default it writes a hidden `<think>…</think>` block before answering. That's powerful for hard problems, but for simple prompts it adds latency and clutters the trace. Append **`/no_think`** to a message to switch reasoning off for that turn (`/think` forces it on).

We use `/no_think` below to keep these first traces fast and clean. A later notebook deliberately turns thinking *on* — watching the reasoning unfold span by span is exactly what tracing is good at.

*(Using a non-reasoning model such as `gemma3:4b` instead? `/no_think` is simply ignored, so the cells below run unchanged.)*

## Make a traced call

The `openai` client is configured with Ollama's local endpoint. The `api_key` is required by the client but ignored by Ollama, so any placeholder works.

In [ ]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen3:8b"  # must match a model you've `ollama pull`-ed

response = client.chat.completions.create(
    model=MODEL,
    # `/no_think` switches off Qwen3's reasoning for this turn (see the note above).
    messages=[{"role": "user", "content": "In one sentence, what is MLflow Tracing? /no_think"}],
)
print(response.choices[0].message.content)

## Look at the trace

Open the UI at <http://127.0.0.1:5001> and click the **Traces** tab (or open the experiment, then *Traces*). You'll see one trace for the call above — expand it to find a span carrying:

- **Inputs:** the `messages` you sent and the model name.
- **Outputs:** the completion text.
- **Latency:** how long the call took.
- **Token usage:** prompt / completion / total tokens.

You wrote no `log_param` or `log_metric` calls — autolog captured all of it. That's the *can't-forget* guarantee, the same idea as `mlflow.sklearn.autolog()` in `ml/a_model_logging`, but for LLM calls.

## Tracing your *own* code with `@mlflow.trace`

Autolog traces the LLM client.   

But your real value is the code *around* the call — prompt construction, retrieval, post-processing.   
Decorate any function with **`@mlflow.trace`** and each call becomes a span; calls nested inside become **child spans**, so the trace mirrors your call tree.

Here a top-level `answer_question` calls `build_prompt` (traced) and the LLM client (autologged) — three spans in one trace.

In [ ]:
@mlflow.trace
def build_prompt(question: str):
    return [
        {"role": "system", "content": "Answer in one concise sentence."},
        {"role": "user", "content": f"{question} /no_think"},
    ]

@mlflow.trace
def answer_question(question: str) -> str:
    messages = build_prompt(question)
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    return resp.choices[0].message.content.strip()

print(answer_question("What is a span in MLflow Tracing?"))

Back in the **Traces** tab, the newest trace is a tree: `answer_question` at the root, with `build_prompt` and the LLM call as children. That nesting is the whole point — for a real chain or agent, the trace shows exactly where time went and what each step saw.

## The one-line swap to the OpenAI API

Everything above ran locally and free. To run the *same* code against the real OpenAI API, change only the client — `mlflow.openai.autolog()` and every `@mlflow.trace` stay exactly as they are:

```python
client = OpenAI()        # default base_url is OpenAI; reads OPENAI_API_KEY from the env
MODEL = "gpt-4o-mini"
```

Set `OPENAI_API_KEY` and it works unchanged. Note this **spends money per call** — Ollama stays the zero-cost default for the rest of this track.

## Optional: tracing a hosted model (Azure OpenAI)

Same lesson, a third backend. If you have an **Azure OpenAI / AI Foundry** endpoint, you reach it with the `AzureOpenAI` client — still the `openai` SDK, so `mlflow.openai.autolog()` (already on) traces it with **no extra setup**. Useful when you want a stronger hosted model than local Ollama but the *same* trace view.

Credentials come from a **gitignored `.env`** at the repo root — never hard-code keys in a notebook:

```
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_BASE_URL=https://<resource>.openai.azure.com/openai
AZURE_OPENAI_API_VERSION=2024-10-21
AZURE_OPENAI_LIGHT_MODEL=gpt-5.4-nano   # one of YOUR deployment names — pick a light/fast tier
```

We deliberately pick a **light/fast deployment** (a `nano`/`mini` tier): cheaper and quicker, which is all a quickstart needs.

In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # read the gitignored .env at the repo root

az = AzureOpenAI(
    # the client re-adds `/openai`, so hand it the resource root
    azure_endpoint=os.environ["AZURE_OPENAI_BASE_URL"].removesuffix("/openai"),
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
AZ_MODEL = os.environ.get("AZURE_OPENAI_LIGHT_MODEL", "gpt-5.4-nano")

# AzureOpenAI is still the openai client, so the autolog from earlier traces this too.
resp = az.chat.completions.create(
    model=AZ_MODEL,  # an Azure *deployment* name, not a base model id
    messages=[{"role": "user", "content": "In one sentence, what is MLflow Tracing?"}],
)
print(resp.choices[0].message.content)

Open the **Traces** tab once more: this call appears just like the Ollama ones — same span shape (inputs / outputs / latency / token usage), different backend. That's the payoff of autolog instrumenting the *client*: swap Ollama ↔ OpenAI ↔ Azure and your tracing code doesn't change at all.

## Next steps

- **`b_tracing_a_multistep_app.ipynb`** — a small RAG / tool-using chain where tracing earns its keep: many nested spans across retrieval and generation.
- **`c_genai_evaluation.ipynb`** — once you can *see* the output, how do you *score* it? `mlflow.genai.evaluate()` with LLM-as-judge scorers — the GenAI analog of `ml/e_model_evaluation`.